In [1]:
! pip install torchtext

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchtext.datasets import WikiText103
from torchtext.data.utils import get_tokenizer
from collections import Counter
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np

# 1. Download & tokenize WikiText-103 train set
train_iter = list(WikiText103(split='train'))  # make it a list
tokenizer = get_tokenizer('basic_english')
tokens = [token for line in train_iter for token in tokenizer(line)]

# 2. Build vocab
counter = Counter(tokens)
vocab = {word: idx + 2 for idx, (word, _) in enumerate(counter.most_common())}
vocab['<unk>'] = 0
vocab['<pad>'] = 1
inv_vocab = {idx: word for word, idx in vocab.items()}

vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")

# 3. Numericalize the tokens
numericalized = [vocab.get(token, vocab['<unk>']) for token in tokens]

# 4. Prepare dataset
seq_len = 30

class LMDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx:idx+self.seq_len], dtype=torch.long)
        y = torch.tensor(self.data[idx+1:idx+self.seq_len+1], dtype=torch.long)
        return x, y

dataset = LMDataset(numericalized, seq_len)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# 5. Custom LSTMCell and LanguageModel
class CustomLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.W_ih = nn.Linear(input_size, 4 * hidden_size)
        self.W_hh = nn.Linear(hidden_size, 4 * hidden_size)

    def forward(self, x, hx):
        h_prev, c_prev = hx
        gates = self.W_ih(x) + self.W_hh(h_prev)
        i, f, g, o = gates.chunk(4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o)
        c = f * c_prev + i * g
        h = o * torch.tanh(c)
        return h, c, {'input_gate': i, 'forget_gate': f, 'candidate': g, 'output_gate': o}

class LanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=1)
        self.lstm_cell = CustomLSTMCell(embed_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hx=None):
        batch_size, seq_len = x.size()
        if hx is None:
            h = torch.zeros(batch_size, self.lstm_cell.hidden_size, device=x.device)
            c = torch.zeros(batch_size, self.lstm_cell.hidden_size, device=x.device)
        else:
            h, c = hx

        outputs = []
        gates_all = []

        for t in range(seq_len):
            emb = self.embedding(x[:, t])
            h, c, gates = self.lstm_cell(emb, (h, c))
            outputs.append(h.unsqueeze(1))
            gates_all.append(gates)

        outputs = torch.cat(outputs, dim=1)
        logits = self.fc(outputs)
        return logits, (h, c), gates_all

# 6. Training setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LanguageModel(vocab_size, embed_dim=128, hidden_dim=256).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# 7. Training loop (1 epoch)
model.train()
for batch_idx, (x_batch, y_batch) in enumerate(dataloader):
    x_batch, y_batch = x_batch.to(device), y_batch.to(device)
    optimizer.zero_grad()
    logits, _, _ = model(x_batch)
    loss = criterion(logits.view(-1, vocab_size), y_batch.view(-1))
    loss.backward()
    optimizer.step()

    if batch_idx % 100 == 0:
        print(f"Batch {batch_idx}, Loss: {loss.item():.4f}")

# 8. Visualization
def plot_gates(gates_all, sample_idx=0):
    input_gates = torch.stack([g['input_gate'][sample_idx] for g in gates_all]).cpu().detach().numpy()
    forget_gates = torch.stack([g['forget_gate'][sample_idx] for g in gates_all]).cpu().detach().numpy()
    output_gates = torch.stack([g['output_gate'][sample_idx] for g in gates_all]).cpu().detach().numpy()
    candidate = torch.stack([g['candidate'][sample_idx] for g in gates_all]).cpu().detach().numpy()

    time_steps = range(len(gates_all))

    plt.figure(figsize=(12,8))
    plt.subplot(4,1,1)
    plt.plot(time_steps, input_gates)
    plt.title('Input Gate')

    plt.subplot(4,1,2)
    plt.plot(time_steps, forget_gates)
    plt.title('Forget Gate')

    plt.subplot(4,1,3)
    plt.plot(time_steps, output_gates)
    plt.title('Output Gate')

    plt.subplot(4,1,4)
    plt.plot(time_steps, candidate)
    plt.title('Candidate Cell State')
    plt.xlabel('Time Step')

    plt.tight_layout()
    plt.show()

# 9. Run inference and plot
model.eval()
sample_seq = numericalized[:seq_len]
input_tensor = torch.tensor(sample_seq, dtype=torch.long).unsqueeze(0).to(device)
with torch.no_grad():
    logits, _, gates_all = model(input_tensor)

plot_gates(gates_all)


AttributeError: 'NoneType' object has no attribute 'Lock'
This exception is thrown by __iter__ of _MemoryCellIterDataPipe(remember_elements=1000, source_datapipe=_ChildDataPipe)